# Model Uncertainty in Climate Projections

This notebook explores CMIP6 multi-model spread for **California** under **SSP3-7.0**, using data retrieved via the Cal-Adapt Analytics Engine. It builds a progression of visualizations that characterize *how much* models disagree, *where* they disagree, and *which variables* carry the most uncertainty.

**Workflow:**
1. Retrieve and preprocess CMIP6 multi-model temperature data
2. Compute annual anomalies relative to the 1850–1900 pre-industrial baseline
3. Visualize model spread: plume chart → ridgeline distributions → spatial diagnostics → warming-level scatter
4. Compare inter-model spread across variables (temperature vs. precipitation)

## Step 0: Setup — imports and configuration

In [ ]:
import warnings

import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import holoviews as hv
import hvplot.xarray
import ridgeplot
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact
import os

from bokeh.models import HoverTool
from bokeh.io import export_png
from bokeh.embed import file_html
from bokeh.resources import INLINE
from bokeh.models import HoverTool, ColumnDataSource, Div
from bokeh.layouts import column

from climakitae.explore.uncertainty import (
    CmipOpt,
    grab_multimodel_data,
    weighted_temporal_mean,
    calc_anom,
    cmip_mmm,
)
from climakitae.core.paths import GWL_1850_1900_FILE
from climakitae.util.utils import read_csv_file
from climakitae.util.colormap import read_ae_colormap

warnings.filterwarnings("ignore")
hv.extension("bokeh")
hv.plotting.bokeh.element.ElementPlot.active_tools = ['pan’]
%config InlineBackend.figure_format = 'retina'

In [ ]:
os.makedirs("figures", exist_ok=True)
os.makedirs("figures/static", exist_ok=True)
os.makedirs("figures/html", exist_ok=True)

### Step 1: Retrieve CMIP6 multi-model data

We retrieve monthly near-surface air temperature (`tas`) for all available CMIP6 models with both historical and SSP3-7.0 simulations, subsetted to California.

In [ ]:
# Configure data options
copt = CmipOpt()
copt.variable = "tas"  # near-surface air temperature
copt.area_subset = "states"
copt.location = "California"
copt.area_average = False  # retain spatial dimensions
copt.timescale = "Amon"  # monthly

In [ ]:
# Retrieve and pre-process all available CMIP6 models
mdls_ds = grab_multimodel_data(copt)
mdls_ds

### Step 2: Compute annual anomalies and multi-model mean

We calculate annual averages, convert to temperature anomalies relative to a 1850–1900 pre-industrial baseline, and compute the multi-model mean (MMM). The anomaly framing places results in a [global warming levels](warming_levels.ipynb) context.

In [ ]:
# Annual average → Celsius → anomaly relative to 1850–1900 → multi-model mean
xy_ds_yr = weighted_temporal_mean(mdls_ds).compute()
xy_ds_yr = xy_ds_yr - 273.15
xy_ds_yr.tas.attrs["units"] = "°C"

xy_anom = calc_anom(xy_ds_yr, base_start=1850, base_end=1900)
xy_anom.tas.attrs["units"] = "°C"

xy_anom_mmm = cmip_mmm(xy_anom)

In [ ]:
# Area-average timeseries; split into historical and future periods
hist_start, hist_end, ssp_end = 1950, 2014, 2100

cmip_anom = xy_anom.groupby("time").mean(dim=["x", "y"])
cmip_anom.tas.attrs["units"] = "°C"
cmip_anom_mmm = cmip_mmm(cmip_anom)

hist_anom = cmip_anom.sel(time=slice(hist_start, hist_end))
hist_anom_mmm = cmip_anom_mmm.sel(time=slice(hist_start, hist_end))
ssp_anom = cmip_anom.sel(time=slice(hist_end, ssp_end))
ssp_anom_mmm = cmip_anom_mmm.sel(time=slice(hist_end, ssp_end))

In [ ]:
# Cal-Adapt: Analytics Engine model subset
cae_mdls_ls = [
    "FGOALS-g3",
    "EC-Earth3-Veg",
    "CESM2",
    "CNRM-ESM2-1",
    "MIROC6",
    "MPI-ESM1-2-HR",
    "EC-Earth3",
    "TaiESM1",
]
cmip_mdls_ls = [
    cmip_anom.simulation.values[i]
    for i in range(len(cmip_anom.simulation.values))
    if cmip_anom.simulation.values[i] not in cae_mdls_ls
]

cae_anom = cmip_anom.sel(simulation=cae_mdls_ls,
                         time=slice(hist_start, ssp_end))

### Step 3: Visualize the model spread

#### Step 3a: Select a warming level of interest

We use the year each model reaches a global warming level to anchor cross-model comparisons on physically comparable climate states, rather than a fixed calendar year.

In [ ]:
# Set warming level — options: 1.5, 2.0, 3.0, 4.0
warm_level = 3.0

In [ ]:
# Identify the year each model reaches warm_level
gwl_times = read_csv_file(GWL_1850_1900_FILE, index_col=[0, 1, 2])

scenario = "ssp370"
sim_idx = []
for simulation in ssp_anom.simulation.values:
    if simulation in gwl_times.index:
        if simulation == "CESM2":
            sim_idx.append((simulation, "r11i1p1f1", scenario))
        elif simulation == "CNRM-ESM2-1":
            sim_idx.append((simulation, "r1i1p1f2", scenario))
        else:
            sim_idx.append((simulation, "r1i1p1f1", scenario))

xy_da_list = []
year_reached_by_sim = []
for i in sim_idx:
    year_str = str(gwl_times[str(warm_level)].loc[i])[:4]
    if len(year_str) != 4:
        print(f"{warm_level}°C warming level not reached for {i[0]}")
    else:
        year_reached_by_sim.append((i, int(year_str)))
        xy_da_list.append(xy_anom.sel(time=int(year_str), simulation=i[0]))

thresh_df = pd.DataFrame(
    data=year_reached_by_sim, columns=[
        "simulation", "year_warming_level_reached"]
)
xy_by_warmlevel = xr.concat(xy_da_list, dim="simulation")

#### Step 3b: Plume chart — full model spread through 2100

A plume chart improves on a spaghetti plot by foregrounding the **10th–90th percentile envelope** (practical planning range), the **multi-model mean** (central estimate), and the **Cal-Adapt AE subset** highlighted within the broader CMIP6 archive.

In [ ]:
# cmip_anom = cmip_anom.sel(time=cmip_anom.time.isin(range(1980, 2100)))
# hist_anom = hist_anom.sel(time=hist_anom.time.isin(range(1980, 2100)))

full = (
    xy_anom.sel(time=xy_anom.time.isin(range(1980, 2100)))
    .groupby("time")
    .mean(dim=["x", "y"])
)
full.tas.attrs["units"] = "°C"
full_mmm = cmip_mmm(full)

In [ ]:
# Stack all model timeseries into (n_models, n_years) for percentile calculation
all_sims = full.simulation.values
all_years = full.time.values

sim_array = np.array([full.sel(simulation=s).tas.values for s in all_sims])

p1 = np.nanpercentile(sim_array, 1, axis=0)
p2 = np.nanpercentile(sim_array, 2, axis=0)
p10 = np.nanpercentile(sim_array, 10, axis=0)
p25 = np.nanpercentile(sim_array, 25, axis=0)
p75 = np.nanpercentile(sim_array, 75, axis=0)
p90 = np.nanpercentile(sim_array, 90, axis=0)
p98 = np.nanpercentile(sim_array, 98, axis=0)
p99 = np.nanpercentile(sim_array, 99, axis=0)
mmm = np.nanmean(sim_array, axis=0)

cae_indices = [i for i, s in enumerate(all_sims) if s in cae_mdls_ls]

In [ ]:
def remove_bokeh_logo(plot, element):
    plot.state.toolbar.logo = None

In [ ]:
cmip_anom = full.sel(simulation=~full.simulation.isin(
    cae_anom.simulation.values))
cae_anom = full.sel(simulation=full.simulation.isin(
    cae_anom.simulation.values))

In [ ]:
lw, alpha = 0.8, 0.30

# ── Hook ──────────────────────────────────────────────────────────────────────


def single_tooltip_hook(plot, element):
    p = plot.state
    p.tools = [t for t in p.tools if not isinstance(t, HoverTool)]

    ssp_data = {"xs": [], "ys": [], "simulation": []}
    cae_data = {"xs": [], "ys": [], "simulation": []}
    ssp_renderers = []
    cae_renderers = []

    for r in p.renderers:
        if not hasattr(r, "glyph") or r.glyph.__class__.__name__ != "Line":
            continue
        if "simulation" not in r.data_source.data:
            continue
        color_val = r.glyph.line_color
        if hasattr(color_val, "value"):
            color_val = color_val.value
        times = list(r.data_source.data["time"])
        tas = list(r.data_source.data["tas"])
        sim = str(r.data_source.data["simulation"][0])

        if color_val == "#4472A8":
            cae_data["xs"].append(times)
            cae_data["ys"].append(tas)
            cae_data["simulation"].append(sim)
            cae_renderers.append(r)
        else:
            ssp_data["xs"].append(times)
            ssp_data["ys"].append(tas)
            ssp_data["simulation"].append(sim)
            ssp_renderers.append(r)

    ssp_src = ColumnDataSource(ssp_data)
    cae_src = ColumnDataSource(cae_data)

    js = CustomJS(
        args=dict(
            ssp_src=ssp_src,
            cae_src=cae_src,
            ssp_renderers=ssp_renderers,
            cae_renderers=cae_renderers,
            base_alpha=alpha,
            base_lw=lw,
        ),
        code="""
        const dataX = cb_obj.x;
        const dataY = cb_obj.y;
        if (dataX === null || dataX === undefined) return;

        if (!window._bkMouseTracking) {
            window._bkMouseTracking = true;
            document.addEventListener('mousemove', (e) => {
                window._bkMouseX = e.clientX;
                window._bkMouseY = e.clientY;
            });
        }

        if (!window._customTip) {
            const tip = document.createElement('div');
            tip.style.cssText = `
                position: fixed; background: white; border: 1px solid #ddd;
                border-radius: 5px; padding: 8px 12px; font-size: 12px;
                font-family: Arial, sans-serif; pointer-events: none; display: none;
                z-index: 99999; box-shadow: 2px 2px 8px rgba(0,0,0,0.15);
                line-height: 1.7; min-width: 200px; max-height: 260px; overflow-y: auto;
            `;
            document.body.appendChild(tip);
            window._customTip = tip;
        }
        const tip = window._customTip;

        const year = Math.round(dataX);
        const Y_THRESHOLD = 0.1;

        const entries = [];
        for (const src of [ssp_src, cae_src]) {
            const allXs = src.data['xs'];
            const allYs = src.data['ys'];
            const sims  = src.data['simulation'];
            for (let i = 0; i < allXs.length; i++) {
                const xs = allXs[i], ys = allYs[i];
                let idx = 0, dxMin = Infinity;
                for (let j = 0; j < xs.length; j++) {
                    const d = Math.abs(xs[j] - dataX);
                    if (d < dxMin) { dxMin = d; idx = j; }
                }
                if (!isNaN(ys[idx]) && Math.abs(ys[idx] - dataY) <= Y_THRESHOLD) {
                    entries.push({ sim: sims[i], y: ys[idx] });
                }
            }
        }

        entries.sort((a, b) => b.y - a.y);
        const seen = new Set();
        const unique = entries.filter(e => {
            if (seen.has(e.sim)) return false;
            seen.add(e.sim);
            return true;
        });

        // If no lines nearby, reset everything and hide tip — no dimming
        if (unique.length === 0) {
            for (const renderers of [ssp_renderers, cae_renderers]) {
                renderers.forEach(r => {
                    r.glyph.line_alpha = base_alpha;
                    r.glyph.line_width = base_lw;
                });
            }
            tip.style.display = 'none';
            return;
        }

        // Only dim/highlight when we have actual hits
        const hitSims = new Set(unique.map(e => e.sim));
        for (const [renderers, src] of [[ssp_renderers, ssp_src], [cae_renderers, cae_src]]) {
            const sims = src.data['simulation'];
            renderers.forEach((r, i) => {
                const isHit = hitSims.has(sims[i]);
                r.glyph.line_alpha = isHit ? 0.85 : base_alpha * 0.4;
                r.glyph.line_width = isHit ? base_lw * 1.8 : base_lw * 0.8;
            });
        }

        if (unique.length === 0) { tip.style.display = 'none'; return; }

        const rows = unique.map(e =>
            `<div style="display:flex;justify-content:space-between;gap:12px;">
                <span style="color:#333">${e.sim}</span>
                <span style="color:#c0392b;font-weight:bold">${e.y.toFixed(2)} \u00b0C</span>
             </div>`
        ).join('');

        tip.innerHTML = rows +
            `<div style="margin-top:6px;padding-top:5px;border-top:1px solid #eee;
                         color:#555;font-size:11px;text-align:right;">
                Year: <b>${year}</b></div>`;
        tip.style.display = 'block';

        const mx = window._bkMouseX || 0;
        const my = window._bkMouseY || 0;
        const tw = tip.offsetWidth || 210, th = tip.offsetHeight || 100;
        let tx = mx + 16, ty = my - th / 2;
        if (tx + tw > window.innerWidth  - 8) tx = mx - tw - 16;
        if (ty < 8) ty = 8;
        if (ty + th > window.innerHeight - 8) ty = window.innerHeight - th - 8;
        tip.style.left = tx + 'px';
        tip.style.top  = ty + 'px';
    """,
    )

    hide_js = CustomJS(
        args=dict(
            ssp_renderers=ssp_renderers,
            cae_renderers=cae_renderers,
            base_alpha=alpha,
            base_lw=lw,
        ),
        code="""
        if (window._customTip) window._customTip.style.display = 'none';
        for (const renderers of [ssp_renderers, cae_renderers]) {
            renderers.forEach(r => {
                r.glyph.line_alpha = base_alpha;
                r.glyph.line_width = base_lw;
            });
        }
    """,
    )

    p.js_on_event("mousemove", js)
    p.js_on_event("mouseleave", hide_js)


# ── Plot elements ─────────────────────────────────────────────────────────────

SSP_COLOR = "#0F5AB1"  # blue for CMIP6
CAE_COLOR = "#A8CAF0"  # blue for Cal-Adapt

SSP_COLOR = "#2166AC"  # blue — CMIP6
CAE_COLOR = "#35978F"  # teal — Cal-Adapt

SSP_COLOR = "#4393C3"  # medium blue — CMIP6
CAE_COLOR = "#762A83"  # deep purple — Cal-Adapt

SSP_COLOR = "#2166AC"  # strong mid-blue — CMIP6 (many lines)
CAE_COLOR = "#D6604D"  # muted red-orange — Cal-Adapt (fewer lines)

band_outer = hv.Area(
    (all_years, p2, p98),
    kdims=["year"],
    vdims=["p2", "p98"],
    label="2nd-98th percentile",
).opts(
    fill_color="#E8956D",
    fill_alpha=0.45,
    line_color=None,
    nonselection_fill_alpha=0.45,
    nonselection_fill_color="#E8956D",
)

all_ssp = cmip_anom.hvplot.line(
    x="time",
    by="simulation",
    line_width=lw,  # line_dash="dotted",
    color=SSP_COLOR,
    legend=False,
    alpha=alpha,
)

all_cae = cae_anom.hvplot.line(
    x="time", by="simulation", line_width=lw, color=CAE_COLOR, legend=False, alpha=alpha
)

mmm = full_mmm.hvplot.line(
    x="time", line_width=lw * 3, color="#1a1a1a", label="Multi-model mean"
).opts(nonselection_line_alpha=1.0, nonselection_line_color="#1a1a1a")

wl_min = int(thresh_df["year_warming_level_reached"].min())
wl_max = int(thresh_df["year_warming_level_reached"].max())
wl_band = hv.VSpan(wl_min, wl_max).opts(
    fill_color="#333",
    fill_alpha=0.07,
    line_color=None,
    nonselection_fill_alpha=0.07,
    nonselection_fill_color="#333",
)

vline = hv.VLine(hist_end).opts(
    color="#555", line_width=0.8, line_dash="dashed", line_alpha=0.6
)
hline = hv.HLine(0).opts(color="#999", line_width=0.6, line_dash="dotted")

nan = [np.nan]
legend_cmip6 = hv.Curve((nan, nan), label="CMIP6 models").opts(
    color=SSP_COLOR,
    line_width=lw * 2,
    nonselection_line_alpha=1.0,
    nonselection_line_color=SSP_COLOR,
)

legend_cae = hv.Curve((nan, nan), label="Cal-Adapt AE models").opts(
    color=CAE_COLOR,
    line_width=lw * 2,
    nonselection_line_alpha=1.0,
    nonselection_line_color=CAE_COLOR,
)

legend_wl = hv.Curve((nan, nan), label=f"Years models reaching {warm_level} C").opts(
    color="#333",
    line_width=8,
    line_alpha=0.25,
    nonselection_line_alpha=0.25,
    nonselection_line_color="#333",
)

legend_ssp = hv.Curve((nan, nan), label=f"SSP3-7.0 start ({hist_end})").opts(
    color="#555",
    line_width=1.5,
    line_dash="dashed",
    line_alpha=0.6,
    nonselection_line_alpha=0.6,
    nonselection_line_color="#555",
)

plot = (
    band_outer
    * wl_band
    * all_ssp
    * all_cae
    * mmm
    * vline
    * hline
    * legend_cmip6
    * legend_cae
    * legend_wl
    * legend_ssp
).opts(
    xlim=(1980, 2100),
    width=900,
    height=420,
    title="CMIP6 California surface temperature projections (SSP3-7.0)",
    xlabel="Year",
    ylabel="Temperature anomaly (C) relative to 1850-1900",
    legend_position="top_left",
    toolbar="above",
    show_grid=False,
    ylim=(hist_anom.tas.min().values - 0.25, ssp_anom.tas.max().values + 0.25),
    hooks=[remove_bokeh_logo, single_tooltip_hook],
)

display(plot)
bokeh_plot = hv.render(plot, backend="bokeh")
export_png(
    bokeh_plot, filename="figures/static/temperature_projections.png", scale_factor=2.0
)
html = file_html(bokeh_plot, resources=INLINE, title="Temperature Projections")

with open("figures/html/temperature_projections.html", "w") as f:
    f.write(html)

#### Step 3c: Ridgeline plot — per-model distributions for a chosen period

Each ridge shows one model's distribution of annual temperature anomalies. Wide, flat ridges indicate high within-model spread; non-overlapping ridges indicate high between-model disagreement. Modify `ridge_start` / `ridge_end` to compare near-term vs. end-of-century.

In [ ]:
# Period for ridgeline — modify to compare near-term vs. end-of-century
ridge_start, ridge_end = 2071, 2100  # end-of-century

##### Approach A — annual anomalies (pre-computed `cae_anom` / `cmip_anom`)

Uses the annually-averaged, area-averaged anomalies already computed in Step 2. Quick to run but limited to annual resolution.

In [ ]:
ridge_start, ridge_end = 2071, 2100

#  separate the two groups before building traces
cae_data = cae_anom.sel(time=slice(ridge_start, ridge_end))
cmip_data = cmip_anom.sel(time=slice(ridge_start, ridge_end))


def build_samples(da):
    samples, labels = [], []
    for m in da.simulation.values:
        vals = da.sel(simulation=m).tas.values
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0:
            samples.append([vals])
            labels.append(m)
    return samples, labels


cae_samples, cae_labels = build_samples(cae_data)
cmip_samples, cmip_labels = build_samples(cmip_data)

cmap = read_ae_colormap(cmap="ae_diverging", cmap_hex=True)
cmap = "plasma"

#  build one figure per group


def make_ridge_fig(samples, labels):
    return ridgeplot.ridgeplot(
        samples=samples,
        labels=labels,
        colorscale=cmap,
        opacity=0.75,
        bandwidth=0.4,
        kde_points=1000,
        spacing=0.25,
        line_width=1.5,
        xpad=1.0,
    )


fig_cae = make_ridge_fig(cae_samples, cae_labels)
fig_cmip = make_ridge_fig(cmip_samples, cmip_labels)

n_cae = len(fig_cae.data)
n_cmip = len(fig_cmip.data)

#  get full x range across BOTH figures
all_x = []
for trace in list(fig_cae.data) + list(fig_cmip.data):
    if trace.x is not None:
        all_x.extend([v for v in trace.x if v is not None and not np.isnan(v)])

x_min = min(all_x) - 0.3
x_max = max(all_x) + 0.3

#  combine into one figure — CMIP first, then CAE
fig = go.Figure()

for trace in fig_cmip.data:
    trace.visible = True
    fig.add_trace(trace)

for trace in fig_cae.data:
    trace.visible = False
    fig.add_trace(trace)

#  dropdown visibility masks
show_cmip = [True] * n_cmip + [False] * n_cae
show_cae = [False] * n_cmip + [True] * n_cae

#  layout helpers


def fig_height(labels, factor=25):
    return factor * len(labels) + 100


def fig_width(labels, factor=25):
    return 600


MARGIN = dict(l=60, r=30, t=100, b=60)

fig.update_layout(
    title=dict(
        text=f"All CMIP6 models — {ridge_start}–{ridge_end} (SSP3-7.0)",
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top",
    ),
    xaxis_title="Temperature anomaly (°C) relative to 1850–1900",
    height=fig_height(cmip_labels, factor=25),
    width=fig_width(cmip_labels, factor=25),
    showlegend=False,
    plot_bgcolor="white",
    margin=MARGIN,
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="All CMIP6 models",
                    method="update",
                    args=[
                        {"visible": show_cmip},
                        {
                            "title.text": f"All CMIP6 models — {ridge_start}–{ridge_end} (SSP3-7.0)",
                            "height": fig_height(cmip_labels, factor=25),
                            "width": fig_width(cmip_labels, factor=25),
                            "margin": MARGIN,
                        },
                    ],
                ),
                dict(
                    label="Cal-Adapt AE models",
                    method="update",
                    args=[
                        {"visible": show_cae},
                        {
                            "title.text": f"Cal-Adapt AE models — {ridge_start}–{ridge_end} (SSP3-7.0)",
                            "height": fig_height(cae_labels, factor=50),
                            "width": fig_width(cae_labels, factor=50),
                            "margin": MARGIN,
                        },
                    ],
                ),
            ],
            direction="down",
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.08,
            yanchor="top",
        )
    ],
)

#  axes
fig.update_xaxes(
    range=[x_min, x_max],
    title_text="Temperature anomaly (°C) relative to 1850–1900",
)

fig.update_yaxes(
    showticklabels=False,
    title_text="Model",
)

fig.show()
fig.write_html("figures/html/ridgeplot.html")
fig.write_image("figures/static/ridgeplot.png", scale=3)

##### Approach B — monthly anomalies (raw `mdls_ds`, custom baseline)

Reloads raw monthly data and subtracts a 1971–2010 reference climatology. Provides finer within-period spread at the cost of a longer compute step (uses Dask).

In [ ]:
# from dask.diagnostics import ProgressBar

# ridge_start, ridge_end = "2071-01-01", "2100-01-01"
# base_start, base_end = "1971-01-01", "2000-01-01"

# data = mdls_ds.sel(time=slice(ridge_start, ridge_end))
# base = mdls_ds.sel(time=slice(base_start, base_end))

# cae_data = data.sel(simulation=cae_mdls_ls)
# cae_base = base.sel(simulation=cae_mdls_ls)
# cmip_mdls_ls_monthly = [
#     str(s) for s in data.simulation.values if s not in cae_data.simulation.values]
# cmip_data = data.sel(simulation=data.simulation.isin(cmip_mdls_ls_monthly))
# cmip_base = base.sel(simulation=data.simulation.isin(cmip_mdls_ls_monthly))

# with ProgressBar():
#     cae_data = cae_data.compute()
#     cmip_data = cmip_data.compute()
#     cae_base = cae_base.compute()
#     cmip_base = cmip_base.compute()

In [ ]:
# cae_data = cae_data-cae_base.mean(dim="time")
# cmip_data = cmip_data-cmip_base.mean(dim="time")

In [ ]:
# def build_samples(da):
#     samples, labels = [], []
#     for m in da.simulation.values:
#         vals = da.sel(simulation=m).tas.values
#         vals = vals[~np.isnan(vals)]
#         if len(vals) > 0:
#             samples.append([vals])
#             labels.append(m)
#     return samples, labels


# cae_samples,  cae_labels = build_samples(cae_data.mean(dim=["x", "y"]))
# cmip_samples, cmip_labels = build_samples(cmip_data.mean(dim=["x", "y"]))

# cmap_div = read_ae_colormap(cmap="ae_diverging", cmap_hex=True)

# #  build one figure per group


# def make_ridge_fig(samples, labels, xpad=1.0):
#     return ridgeplot.ridgeplot(
#         samples=samples,
#         labels=labels,
#         colorscale=cmap_div,
#         opacity=0.75,
#         bandwidth=1.0,
#         kde_points=1000,
#         spacing=0.25,
#         line_width=1.5,
#         # xpad=xpad,
#     )


# fig_cae = make_ridge_fig(cae_samples,  cae_labels,  xpad=1.5)
# fig_cmip = make_ridge_fig(cmip_samples, cmip_labels, xpad=1.0)

# n_cae = len(fig_cae.data)
# n_cmip = len(fig_cmip.data)

# #  compute x range from raw sample values + generous padding
# all_raw = []
# for group in cae_samples + cmip_samples:
#     for arr in group:
#         all_raw.extend(arr.tolist())

# data_min = min(all_raw)
# data_max = max(all_raw)
# pad = (data_max - data_min) * 0.3   # 30% of data range on each side

# x_min = data_min - pad
# x_max = data_max + pad

# #  get full y range across both figures
# all_y = []
# for trace in list(fig_cae.data) + list(fig_cmip.data):
#     if trace.y is not None:
#         all_y.extend([v for v in trace.y if v is not None and not np.isnan(v)])

# y_min = min(all_y) - 1.0
# y_max = max(all_y) + 1.0

# #  combine into one figure
# fig = go.Figure()

# for trace in fig_cae.data:
#     trace.visible = True
#     fig.add_trace(trace)

# for trace in fig_cmip.data:
#     trace.visible = False
#     fig.add_trace(trace)

# #  dropdown visibility masks
# show_cae = [True] * n_cae + [False] * n_cmip
# show_cmip = [False] * n_cae + [True] * n_cmip

# #  layout helpers


# def fig_height(labels, factor=25): return factor * len(labels) + 100
# def fig_width(labels,  factor=25): return 600


# MARGIN = dict(l=60, r=30, t=100, b=60)

# fig.update_layout(
#     title=dict(
#         text=f"Cal-Adapt AE models — {ridge_start}–{ridge_end} (SSP3-7.0)",
#         x=0.5,
#         xanchor="center",
#         y=0.98,
#         yanchor="top",
#     ),
#     height=fig_height(cae_labels, factor=50),
#     width=fig_width(cae_labels,   factor=50),
#     showlegend=False,
#     plot_bgcolor="white",
#     margin=MARGIN,
#     updatemenus=[
#         dict(
#             buttons=[
#                 dict(
#                     label="Cal-Adapt AE models",
#                     method="update",
#                     args=[
#                         {"visible": show_cae},
#                         {
#                             "title.text":  f"Cal-Adapt AE models — {ridge_start}–{ridge_end} (SSP3-7.0)",
#                             "height":      fig_height(cae_labels,  factor=50),
#                             "width":       fig_width(cae_labels,   factor=50),
#                             "margin":      MARGIN,
#                             "xaxis.range": [x_min, x_max],
#                             # "yaxis.range": [y_min, y_max],
#                         },
#                     ],
#                 ),
#                 dict(
#                     label="All CMIP6 models",
#                     method="update",
#                     args=[
#                         {"visible": show_cmip},
#                         {
#                             "title.text":  f"All CMIP6 models — {ridge_start}–{ridge_end} (SSP3-7.0)",
#                             "height":      fig_height(cmip_labels, factor=25),
#                             "width":       fig_width(cmip_labels,  factor=25),
#                             "margin":      MARGIN,
#                             "xaxis.range": [x_min, x_max],
#                             # "yaxis.range": [y_min, y_max],
#                         },
#                     ],
#                 ),
#             ],
#             direction="down",
#             showactive=True,
#             x=0.0,
#             xanchor="left",
#             y=1.08,
#             yanchor="top",
#         )
#     ],
# )

# #  axes
# fig.update_xaxes(
#     range=[x_min, x_max],
#     title_text="Temperature anomaly (°C) relative to 1850–1900",
# )

# fig.update_yaxes(
#     showticklabels=False,
#     title_text="Model",
# )

# fig.show()
# fig.write_html("../uncertainty/figures/html/ridgeplot.html")
# fig.write_image("../uncertainty/figures/static/ridgeplot.png", scale=3)

Dashed lines mark each model's **median**. Switch to the near-term period `(2021, 2050)` and rerun — the ridges converge substantially, reflecting lower model uncertainty at shorter time horizons.

#### Step 3e: Spatial uncertainty maps — range, sign agreement, and signal-to-noise

Three complementary diagnostics characterize *where* models agree and where they don't:

- **Range** (max − min): raw spread in absolute terms
- **Sign-agreement** (% of models agreeing on direction of change vs. baseline): the IPCC AR6 standard robustness criterion — regions below 80% agreement should not be used to draw conclusions about the direction of change
- **Signal-to-noise ratio** (MMM ÷ inter-model std): SNR > 1 means the forced signal exceeds model disagreement; SNR < 1 means the projection is not distinguishable from model noise

In [ ]:
# Compute cross-model statistics on the spatial fields at the warming level
mean_data = xy_by_warmlevel.tas.mean(dim="simulation")
std_data = xy_by_warmlevel.tas.std(dim="simulation")
min_data = xy_by_warmlevel.tas.min(dim="simulation")
max_data = xy_by_warmlevel.tas.max(dim="simulation")
range_data = max_data - min_data

# Sign-agreement: fraction of models projecting warming (anomaly > 0)
# xy_by_warmlevel is already an anomaly relative to 1850–1900, so > 0 means warmer
n_models = xy_by_warmlevel.sizes["simulation"]
n_positive = (xy_by_warmlevel.tas > 0).sum(dim="simulation")
# percent of models agreeing on warming
sign_agree = (n_positive / n_models) * 100

# Signal-to-noise ratio: MMM / inter-model std
# Clip std at a small floor to avoid division by zero in uniform regions
snr = mean_data / std_data.where(std_data > 0.01, other=0.01)

In [ ]:
cmap_div = read_ae_colormap(cmap="ae_diverging", cmap_hex=True)
cmap_blue = read_ae_colormap(cmap="ae_blue", cmap_hex=True)

range_vmax = np.nanpercentile(range_data.values, 99)
snr_vmax = np.nanpercentile(np.abs(snr.values), 99)


def make_map(data, title, vmin, vmax, cmap, width=200, height=200):
    return hv.QuadMesh((data["x"], data["y"], data)).opts(
        tools=["hover"],
        colorbar=True,
        cmap=cmap,
        clim=(vmin, vmax),
        xaxis=None,
        yaxis=None,
        clabel="Air Temperature (°C)",
        title=title,
        width=width,
        height=height,
    )

In [ ]:
stat_plots = (
    make_map(mean_data, "Mean anomaly (°C)", 0, 4, cmap=cmap_blue)
    + make_map(range_data, "Range — max minus min (°C)",
               0, range_vmax, cmap=cmap_div)
    + make_map(sign_agree, "Sign agreement (% warming)",
               0, 100, cmap=cmap_blue)
    + make_map(snr, "Signal-to-noise ratio", 0, snr_vmax, cmap=cmap_blue)
)

(
    stat_plots.cols(2)
    .opts(hv.opts.Layout(merge_tools=True, toolbar="below"))
    .opts(title=f"Spatial model uncertainty diagnostics at {warm_level}°C warming")
)

**Reading the maps:**
- **Range**: large values flag where individual model choice matters most for absolute outcomes.
- **Sign agreement**: regions below ~80% (lighter shading) are where models do not even agree on the *direction* of change — projections there should be interpreted with particular caution.
- **SNR > 1** (darker shading): the forced warming signal is distinguishable from model disagreement. SNR < 1 means the projection is dominated by inter-model noise rather than a coherent forced signal.

Coastal California typically has higher sign agreement and SNR than the interior, reflecting the moderating influence of the ocean on both the climate signal and model spread.

#### Step 3d: Regional dot plot — model spread by sub-region

Each dot is one model's temperature anomaly for a California sub-region at a chosen global warming level. The diamond marks the multi-model mean; the grey shadow spans the full model range. Use the dropdown to switch between warming levels. Hover for model names.

In [ ]:
# region definitions
REGIONS = {
    "All California": {"lat": (32.5, 42.0), "lon": (-124.5, -114.0)},
    "Coastal CA": {"lat": (34.0, 38.5), "lon": (-122.5, -119.5)},
    "Sierra Nevada": {"lat": (36.5, 41.0), "lon": (-120.5, -117.5)},
    "Central Valley": {"lat": (35.0, 40.0), "lon": (-122.0, -119.0)},
    "Southern CA": {"lat": (32.5, 35.0), "lon": (-120.5, -115.5)},
}

# pre-compute for all warming levels


def build_warmlevel_data(ds, wl):
    da_list = []
    for sim in ds.simulation:
        sim_name = sim.item()  # extract plain string from DataArray

        # gwl_times has a MultiIndex (simulation, ensemble, scenario)
        # build the correct tuple key matching the notebook's sim_idx logic
        if sim_name == "CESM2":
            key = (sim_name, "r11i1p1f1", "ssp370")
        elif sim_name == "CNRM-ESM2-1":
            key = (sim_name, "r1i1p1f2", "ssp370")
        else:
            key = (sim_name, "r1i1p1f1", "ssp370")

        if key not in gwl_times.index:
            continue

        year_str = str(gwl_times[str(wl)].loc[key])[:4]
        if len(year_str) == 4:
            da_list.append(ds.sel(time=int(year_str), simulation=sim_name))

    if not da_list:
        return None
    return xr.concat(da_list, dim="simulation")


xy_cae_anom = xy_anom.sel(simulation=cae_mdls_ls,
                          time=slice(hist_start, ssp_end))
print("Pre-computing warming level data...")
warmlevel_cache = {wl: build_warmlevel_data(
    xy_anom, wl) for wl in [1.5, 2.0, 2.5, 3.0]}
print("Done.")

In [ ]:
def region_mean(da, lat_range, lon_range):
    return da.sel(y=slice(*lat_range), x=slice(*lon_range)).mean(dim=["x", "y"])

In [ ]:
#  compute global x range across all warming levels and regions
all_vals = []
for wl_data in warmlevel_cache.values():
    if wl_data is None:
        continue
    for region_name, bounds in REGIONS.items():
        if region_name == "All California":
            continue
        for m in wl_data.simulation.values:
            v = float(
                region_mean(
                    wl_data.tas.sel(simulation=m), bounds["lat"], bounds["lon"]
                ).values
            )
            all_vals.append(v)

x_min = min(all_vals) - 0.2
x_max = max(all_vals) + 0.2

In [ ]:
REGION_COLORS = {
    "Coastal CA": "#4472A8",
    "Sierra Nevada": "#C0664A",
    "Central Valley": "#5A9E6F",
    "Southern CA": "#C8A83A",
}

#  plot function


def make_figure(wl):
    xy = warmlevel_cache[wl]
    if xy is None:
        print(f"No models reached {wl}°C.")
        return

    models = list(xy.simulation.values)
    n_regions = len([r for r in REGIONS if r != "All California"])

    fig = go.Figure()

    for ri, (region_name, bounds) in enumerate(
        [(r, b) for r, b in REGIONS.items() if r != "All California"]
    ):
        vals = [
            float(
                region_mean(
                    xy.tas.sel(simulation=m), bounds["lat"], bounds["lon"]
                ).values
            )
            for m in models
        ]
        mmm = float(np.mean(vals))
        jitter = np.linspace(-0.18, 0.18, len(models))

        # grey shadow spanning min to max
        fig.add_shape(
            type="rect",
            x0=min(vals),
            x1=max(vals),
            y0=ri - 0.35,
            y1=ri + 0.35,
            fillcolor="rgba(180,180,180,0.18)",
            line=dict(width=0),
            layer="below",
        )

        # individual model dots
        fig.add_trace(
            go.Scatter(
                x=vals,
                y=[ri + j for j in jitter],
                mode="markers",
                name=region_name,
                legendgroup=region_name,
                marker=dict(
                    color=REGION_COLORS[region_name],
                    size=10,
                    opacity=0.85,
                    line=dict(color="#000000", width=0.5),
                ),
                text=models,
                hovertemplate=(
                    "<b>%{text}</b><br>"
                    f"{region_name}<br>"
                    "Anomaly: +%{x:.2f}°C<extra></extra>"
                ),
            )
        )

        # MMM diamond
        fig.add_trace(
            go.Scatter(
                x=[mmm],
                y=[ri],
                mode="markers",
                name=f"{region_name} MMM",
                legendgroup=region_name,
                showlegend=False,
                marker=dict(
                    color="black",
                    size=8,
                    opacity=0.75,
                    symbol="diamond",
                ),
                hovertemplate=(
                    f"<b>{region_name} — multi-model mean</b><br>"
                    f"+{mmm:.2f}°C<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        width=720,
        height=450,
        margin=dict(l=20, r=150, t=80, b=60),
        plot_bgcolor="white",
        paper_bgcolor="white",
        title=dict(
            text=(
                f"Model spread by region  ·  {wl}°C global warming (SSP3-7.0)<br>"
                "<sup>Dots = individual models · Diamond = multi-model mean · "
                "Shadow = full model range · Click legend to toggle</sup>"
            ),
            font=dict(size=13),
        ),
        xaxis=dict(
            title="Temperature anomaly (°C) relative to 1850–1900",
            range=[x_min, x_max],
            showgrid=True,
            gridcolor="#eeeeee",
            zeroline=False,
            showline=False,
            fixedrange=True,
        ),
        yaxis=dict(
            tickmode="array",
            tickvals=list(range(n_regions)),
            ticktext=[r for r in REGIONS if r != "All California"],
            showgrid=False,
            gridcolor="#eeeeee",
            range=[-0.4, n_regions - 0.4],
            showline=False,
            zeroline=False,
        ),
        legend=dict(
            orientation="v",
            yanchor="middle",
            y=0.5,
            xanchor="left",
            x=1.02,
            font=dict(size=10),
            itemclick="toggle",
            itemdoubleclick="toggleothers",
        ),
    )

    return fig


_ = interact(
    make_figure,
    wl=widgets.Dropdown(
        options=[1.5, 2.0, 2.5, 3.0],
        value=1.5,
        description="Warming (°C):",
        style={"description_width": "100px"},
    ),
)

In [ ]:
#  build one figure per warming level
wl_options = [1.5, 2.0, 2.5, 3.0]
all_traces = []
all_shapes = []
n_traces_per_wl = []
titles = []

for wl in wl_options:
    fig_wl = make_figure(wl)
    n_traces_per_wl.append(len(fig_wl.data))
    titles.append(
        f"Model spread by region  ·  {wl}°C global warming (SSP3-7.0)")
    all_traces.append(fig_wl.data)
    # all_shapes.append(list(fig_wl.layout.shapes))
    all_shapes.append([s.to_plotly_json() for s in fig_wl.layout.shapes])


#  combine into one figure
fig_combined = go.Figure()

for i, (traces, wl) in enumerate(zip(all_traces, wl_options)):
    for trace in traces:
        trace.visible = i == 0
        fig_combined.add_trace(trace)

#  dropdown visibility masks
buttons = []
idx = 0
for i, (wl, n) in enumerate(zip(wl_options, n_traces_per_wl)):
    visible = [False] * sum(n_traces_per_wl)
    for j in range(n):
        visible[idx + j] = True
    buttons.append(
        dict(
            label=f"{wl}°C",
            method="update",
            args=[
                {"visible": visible},
                {
                    "title.text": (
                        f"Model spread by region  ·  {wl}°C global warming (SSP3-7.0)<br>"
                        "<sup>Dots = individual models · Diamond = multi-model mean · "
                        "Shadow = full model range · Click legend to toggle</sup>"
                    ),
                    "shapes": all_shapes[i],
                },
            ],
        )
    )
    idx += n

# use layout from first figure as base
base_fig = make_figure(wl_options[0])
fig_combined.update_layout(base_fig.layout)

#  override with dropdown + annotation
fig_combined.update_layout(
    shapes=all_shapes[0],
    margin=dict(l=20, r=150, t=120, b=60),
    title=dict(
        text=(
            f"Model spread by region  ·  {wl_options[0]}°C global warming (SSP3-7.0)<br>"
            "<sup>Dots = individual models · Diamond = multi-model mean · "
            "Shadow = full model range · Click legend to toggle</sup>"
        ),
        font=dict(size=13),
        y=0.85,
        x=0.5,
        xanchor="center",
        yanchor="top",
    ),
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.22,
            xanchor="left",
            y=1.07,
            yanchor="top",
        )
    ],
)

fig_combined.add_annotation(
    text="Warming Level:",
    x=0.0,
    y=1.05,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=12),
    xanchor="left",
    yanchor="top",
)

fig_combined.show()
fig_combined.write_html("figures/html/region_dots.html")
fig_combined.write_image("figures/static/region_dots.png", scale=3)

### Step 4: When does model uncertainty matter most?

#### Step 4a: Warming-level scatter — global vs. regional response

Models that warm globally faster (reaching the warming level sooner) do not necessarily project proportionally more California warming. Each point is one model; the vertical spread at any x-position is pure model uncertainty in the California regional response.

In [ ]:
WARM_LEVELS = [1.5, 2.0, 3.0, 4.0]

# Combine all simulations for lookup
all_anom = xr.concat([cmip_anom, cae_anom], dim="simulation")


def build_scatter_df(wl):
    scenario = "ssp370"
    sim_idx_wl = []

    # Iterate over ALL simulations (cmip + cae)
    for simulation in all_anom.simulation.values:
        if simulation in gwl_times.index:
            if simulation == "CESM2":
                sim_idx_wl.append((simulation, "r11i1p1f1", scenario))
            elif simulation == "CNRM-ESM2-1":
                sim_idx_wl.append((simulation, "r1i1p1f2", scenario))
            else:
                sim_idx_wl.append((simulation, "r1i1p1f1", scenario))

    year_reached = []
    for i in sim_idx_wl:
        year_str = str(gwl_times[str(wl)].loc[i])[:4]
        if len(year_str) == 4:
            year_reached.append((i, int(year_str)))

    # Use combined dataset for end-of-century anomaly
    eoc_anom = all_anom.sel(time=slice(2071, 2100)).mean(dim="time")

    rows = []
    for sim_tuple, yr in year_reached:
        model = sim_tuple[0]
        try:
            ca_val = float(eoc_anom.sel(simulation=model).tas.values)
            # rows.append({"model": model, "year_wl": yr, "ca_anom": ca_val})
            rows.append(
                {
                    "model": model,
                    "year_wl": yr,
                    "ca_anom": ca_val,
                    "ca_anom_str": f"{ca_val:.2f}",  # pre-formatted string
                }
            )
        except Exception:
            pass

    return pd.DataFrame(rows)


print("Building scatter data for all warming levels...")
scatter_cache = {wl: build_scatter_df(wl) for wl in WARM_LEVELS}
print("Done.")

# Verify CAE models are now included
for wl, df in scatter_cache.items():
    cae_found = [m for m in cae_mdls_ls if m in df["model"].values]
    print(f"WL {wl}: {len(df)} models, CAE found: {cae_found}")

In [ ]:
WL_COLORS = {
    1.5: "#4472A8",
    2.0: "#5A9E6F",
    3.0: "#C8A83A",
    4.0: "#C0664A",
}

fig, ax = plt.subplots(figsize=(8, 4.5))

for wl, color in WL_COLORS.items():
    df = scatter_cache[wl]
    if df.empty:
        continue

    cae_df = df[df["model"].isin(cae_mdls_ls)]
    other_df = df[~df["model"].isin(cae_mdls_ls)]

    # Cal-Adapt AE models — filled circle
    ax.scatter(
        cae_df["year_wl"],
        cae_df["ca_anom"],
        s=70,
        color=color,
        marker="o",
        edgecolors="white",
        linewidths=0.8,
        zorder=4,
        label=f"{wl}°C — Cal-Adapt AE",
    )

    # Other CMIP6 — hollow triangle
    ax.scatter(
        other_df["year_wl"],
        other_df["ca_anom"],
        s=50,
        color=color,
        marker="o",
        facecolors="none",
        edgecolors=color,
        linewidths=1.2,
        zorder=3,
        label=f"{wl}°C — Other CMIP6",
    )

ax.set_xlabel("Year model reaches warming level", fontsize=11)
ax.set_ylabel(
    "California temperature anomaly\n2071–2100 mean (°C)", fontsize=11)
ax.set_title(
    "Global warming rate vs. California end-of-century anomaly\n"
    "by warming level — CMIP6 models (SSP3-7.0)",
    fontsize=11,
    pad=10,
)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=10)
ax.grid(True, alpha=0.2, lw=0.6)

#  custom legend: two sections — warming level colours + marker types
wl_handles = [
    mlines.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=c,
        markersize=8,
        markeredgecolor="white",
        label=f"{wl}°C",
    )
    for wl, c in WL_COLORS.items()
]
marker_handles = [
    mlines.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#888",
        markersize=8,
        markeredgecolor="white",
        label="Cal-Adapt AE",
    ),
    mlines.Line2D(
        [0],
        [0],
        marker="^",
        color="w",
        markerfacecolor="none",
        markersize=8,
        markeredgecolor="#888",
        markeredgewidth=1.2,
        label="Other CMIP6",
    ),
]

leg1 = ax.legend(
    handles=wl_handles,
    title="Warming level",
    title_fontsize=9,
    loc="lower left",
    fontsize=9,
    framealpha=0.9,
)
ax.add_artist(leg1)
ax.legend(
    handles=marker_handles,
    title_fontsize=9,
    loc="lower right",
    fontsize=9,
    framealpha=0.9,
)

plt.tight_layout()

In [ ]:
fig = go.Figure()

for wl, color in WL_COLORS.items():
    df = scatter_cache[wl]
    if df.empty:
        continue

    cae_df = df[df["model"].isin(cae_mdls_ls)]
    other_df = df[~df["model"].isin(cae_mdls_ls)]

    # Cal-Adapt AE — filled circle
    fig.add_trace(
        go.Scatter(
            x=cae_df["year_wl"],
            y=cae_df["ca_anom"],
            mode="markers",
            name=f"{wl}°C — Cal-Adapt AE",
            legendgroup=str(wl),
            marker=dict(
                color=color,
                size=10,
                symbol="circle",
                line=dict(color="white", width=1),
            ),
            text=cae_df["model"],
            hovertemplate=("<b>%{text}</b><br>" "Year reached: %{x}<br>"),
        )
    )

    # Other CMIP6 — hollow circle
    fig.add_trace(
        go.Scatter(
            x=other_df["year_wl"],
            y=other_df["ca_anom"],
            mode="markers",
            name=f"{wl}°C — Other CMIP6",
            legendgroup=str(wl),
            marker=dict(
                color="rgba(0,0,0,0)",  # transparent fill
                size=8,
                symbol="circle",
                line=dict(color=color, width=1.5),
            ),
            text=other_df["model"],
            hovertemplate=("<b>%{text}</b><br>" "Year reached: %{x}<br>"),
        )
    )

fig.update_layout(
    width=780,
    height=400,
    margin=dict(l=60, r=20, t=80, b=60),
    plot_bgcolor="white",
    paper_bgcolor="white",
    title=dict(
        text=(
            "Global warming rate vs. California end-of-century anomaly<br>"
            "<sup>by warming level — CMIP6 models (SSP3-7.0)</sup>"
        ),
        font=dict(size=13),
    ),
    xaxis=dict(
        title="Year model reaches warming level",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
    ),
    yaxis=dict(
        title="California temperature anomaly<br>2071–2100 mean (°C)",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
    ),
    legend=dict(
        orientation="v",
        yanchor="bottom",
        y=0,
        xanchor="left",
        x=1.02,
        font=dict(size=10),
        tracegroupgap=8,
        itemclick="toggle",
        itemdoubleclick="toggleothers",
    ),
)

fig.show()
fig.write_html("../uncertainty/figures/html/wl_scatter.html")
fig.write_image("../uncertainty/figures/static/wl_scatter.png", scale=2)

In [ ]:
import plotly.graph_objects as go
import plotly.colors as pc

all_models = sorted(
    set(m for df in scatter_cache.values() for m in df["model"].unique())
)

palette = pc.qualitative.Alphabet
model_colors = {m: palette[i % len(palette)] for i, m in enumerate(all_models)}

fig = go.Figure()
seen_models = set()

for wl in sorted(scatter_cache.keys()):
    df = scatter_cache[wl]
    if df.empty:
        continue

    for _, row in df.iterrows():
        model = row["model"]
        is_cae = model in cae_mdls_ls
        color = model_colors[model]
        show_legend = model not in seen_models
        seen_models.add(model)

        fig.add_trace(
            go.Scatter(
                x=[row["year_wl"]],
                y=[f"{wl}°C"],
                mode="markers",
                name=model,
                legendgroup=model,
                showlegend=False,  # no legend — hover shows the name
                marker=dict(
                    color=color,
                    size=14,
                    symbol="circle",  # filled for all
                    line=dict(
                        color="white" if is_cae else color,
                        width=2 if is_cae else 0,
                    ),
                ),
                customdata=[[model, wl, row["ca_anom"]]],
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Warming level: %{customdata[1]}°C<br>"
                    "Year reached: %{x}<br>"
                    "CA anomaly: %{customdata[2]:.2f}°C"
                    "<extra></extra>"
                ),
            )
        )

fig.update_layout(
    width=650,
    height=350,
    margin=dict(l=80, r=40, t=80, b=60),
    plot_bgcolor="white",
    paper_bgcolor="white",
    title=dict(
        text=(
            "Global warming rate by model and warming level<br>"
            "<sup>Year each CMIP6 model reaches each warming level (SSP3-7.0)"
        ),
        font=dict(size=13),
    ),
    xaxis=dict(
        title="Year model reaches warming level",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
    ),
    yaxis=dict(
        title="Global warming level",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
        categoryorder="array",
        categoryarray=[f"{wl}°C" for wl in sorted(scatter_cache.keys())],
    ),
    showlegend=False,
)

fig.show()
fig.write_html("figures/html/wl_scatter.html")
fig.write_image("figures/static/wl_scatter.png", scale=2)

In [ ]:
import plotly.graph_objects as go
import numpy as np

np.random.seed(42)

CMIP_COLOR = "#2166AC"

fig = go.Figure()

wls = sorted(scatter_cache.keys())
wl_to_y = {wl: i for i, wl in enumerate(wls)}

for wl in wls:
    df = scatter_cache[wl].copy()
    if df.empty:
        continue

    y_base = wl_to_y[wl]
    jitter = 0.12

    fig.add_trace(
        go.Scatter(
            x=df["year_wl"],
            y=y_base + np.random.uniform(-jitter, jitter, len(df)),
            mode="markers",
            name="CMIP6 models",
            legendgroup="cmip6",
            showlegend=(wl == wls[0]),
            marker=dict(color=CMIP_COLOR, size=9, opacity=0.7),
            customdata=list(zip(df["model"], [wl] * len(df), df["ca_anom"])),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Warming level: %{customdata[1]}°C<br>"
                "Year reached: %{x}<br>"
                "CA anomaly: %{customdata[2]:.2f}°C"
                "<extra></extra>"
            ),
        )
    )

fig.update_layout(
    width=750,
    height=380,
    margin=dict(l=80, r=40, t=80, b=60),
    plot_bgcolor="white",
    paper_bgcolor="white",
    title=dict(
        text=(
            "Global warming rate by model and warming level<br>"
            "<sup>Year each CMIP6 model reaches each warming level (SSP3-7.0)</sup>"
        ),
        font=dict(size=13),
    ),
    xaxis=dict(
        title="Year model reaches warming level",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
    ),
    yaxis=dict(
        title="Global warming level",
        tickmode="array",
        tickvals=list(wl_to_y.values()),
        ticktext=[f"{wl}°C" for wl in wls],
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
        range=[-0.5, len(wls) - 0.5],
    ),
    showlegend=False,
)

fig.show()
fig.write_html("figures/html/wl_scatter.html")
fig.write_image("figures/static/wl_scatter.png", scale=2)

In [ ]:
all_anom = xr.concat([ssp_anom, cae_anom], dim="simulation")

In [ ]:
import plotly.graph_objects as go
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

np.random.seed(42)

# ── Color gradient by mean CA anomaly across all WLs ─────────────────────────
# Compute each model's mean ca_anom across all WLs for coloring
model_mean_anom = pd.concat(scatter_cache.values()).groupby("model")[
    "ca_anom"].mean()
vmin = model_mean_anom.min()
vmax = model_mean_anom.max()

# cold=blue, warm=red — swap to cmap_div if defined
# Build a matplotlib colormap from cmap_div
cmap = mcolors.LinearSegmentedColormap.from_list("cmap_div", cmap_div[1:-1])
plotly_colorscale = [
    [i / (len(cmap_div[1:-1]) - 1), c] for i, c in enumerate(cmap_div[1:-1])
]

model_mean_anom = pd.concat(scatter_cache.values()).groupby("model")[
    "ca_anom"].mean()
vmin = model_mean_anom.min()
vmax = model_mean_anom.max()
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)


def model_color(model):
    val = model_mean_anom.get(model, (vmin + vmax) / 2)
    rgba = cmap(norm(val))
    return f"rgba({int(rgba[0]*255)},{int(rgba[1]*255)},{int(rgba[2]*255)},1)"


fig = go.Figure()
wls = sorted(scatter_cache.keys())
wl_to_y = {wl: i for i, wl in enumerate(wls)}
cae_set = set(m.strip() for m in cae_mdls_ls)

for wl in wls:
    df = scatter_cache[wl].copy()
    if df.empty:
        continue

    df["model"] = df["model"].astype(str)
    cae_df = df[df["model"].isin(cae_set)]
    other_df = df[~df["model"].isin(cae_set)]

    y_base = wl_to_y[wl]
    jitter = 0.12

    # Other CMIP6 — circles
    if len(other_df) > 0:
        fig.add_trace(
            go.Scatter(
                x=other_df["year_wl"],
                y=y_base + np.random.uniform(-jitter, jitter, len(other_df)),
                mode="markers",
                name="Other CMIP6",
                legendgroup="cmip6",
                showlegend=(wl == wls[0]),
                marker=dict(
                    symbol="circle",
                    color=[model_color(m) for m in other_df["model"]],
                    size=10,
                    line=dict(color="silver", width=1.0),
                ),
                customdata=list(
                    zip(
                        other_df["model"],
                        [wl] * len(other_df),
                        [
                            f"{v:.2f}" if not np.isnan(v) else "N/A"
                            for v in other_df["ca_anom"]
                        ],
                    )
                ),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Warming level: %{customdata[1]}°C<br>"
                    "Year reached: %{x}<br>"
                    "CA anomaly: %{customdata[2]}°C"
                    "<extra></extra>"
                ),
            )
        )

    # Cal-Adapt AE — diamonds
    if len(cae_df) > 0:
        fig.add_trace(
            go.Scatter(
                x=cae_df["year_wl"],
                y=y_base + np.random.uniform(-jitter, jitter, len(cae_df)),
                mode="markers",
                name="Cal-Adapt AE",
                legendgroup="cae",
                showlegend=(wl == wls[0]),
                marker=dict(
                    symbol="diamond",
                    color=[model_color(m) for m in cae_df["model"]],
                    size=10,
                    line=dict(color="silver", width=1.0),
                ),
                customdata=list(
                    zip(
                        cae_df["model"],
                        [wl] * len(cae_df),
                        [
                            f"{v:.2f}" if not np.isnan(v) else "N/A"
                            for v in cae_df["ca_anom"]
                        ],
                    )
                ),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Warming level: %{customdata[1]}°C<br>"
                    "Year reached: %{x}<br>"
                    "CA anomaly: %{customdata[2]}°C"
                    "<extra></extra>"
                ),
            )
        )

# Colorbar dummy trace
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale=plotly_colorscale,
            cmin=vmin,
            cmax=vmax,
            color=[vmin, vmax],
            colorbar=dict(
                title=dict(
                    text="Mean CA<br>anomaly (°C)", side="right", font=dict(size=10)
                ),
                thickness=12,
                len=0.8,
                tickfont=dict(size=9),
            ),
            showscale=True,
            size=0,
        ),
        showlegend=False,
        hoverinfo="skip",
    )
)

fig.update_layout(
    width=700,
    height=400,
    margin=dict(l=80, r=120, t=80, b=60),
    plot_bgcolor="white",
    paper_bgcolor="white",
    title=dict(
        text=(
            "Global warming rate by model and warming level<br>"
            "<sup>Year each CMIP6 model reaches each warming level (SSP3-7.0)</sup>"
        ),
        font=dict(size=13),
    ),
    xaxis=dict(
        title="Year model reaches warming level",
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
    ),
    yaxis=dict(
        title="Global warming level",
        tickmode="array",
        tickvals=list(wl_to_y.values()),
        ticktext=[f"{wl}°C" for wl in wls],
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
        showline=False,
        range=[-0.5, len(wls) - 0.5],
    ),
    legend=dict(
        orientation="v",
        yanchor="bottom",
        y=0.02,
        xanchor="left",
        x=1.12,
        font=dict(size=10),
    ),
)

fig.show()
fig.write_html("figures/html/wl_scatter.html")
fig.write_image("figures/static/wl_scatter.png", scale=2)

### Step 5: Inter-model spread across variables

Model uncertainty is not the same for all variables. Temperature projections have relatively low inter-model spread (high SNR), while precipitation is far more uncertain — models often disagree on both the magnitude *and* the sign of future changes. This section makes that contrast concrete using the same CMIP6 archive.

#### Step 5a: Retrieve precipitation data

In [ ]:
# Retrieve CMIP6 precipitation (pr) — same spatial domain and models as temperature
copt_pr = CmipOpt()
copt_pr.variable = "pr"  # precipitation
copt_pr.area_subset = "states"
copt_pr.location = "California"
copt_pr.area_average = False
copt_pr.timescale = "Amon"

mdls_pr = grab_multimodel_data(copt_pr)

In [ ]:
# Annual average → convert to mm/day → anomaly relative to 1981–2010 climatology
pr_yr = weighted_temporal_mean(mdls_pr).compute()
pr_yr = pr_yr * 86400  # kg m-2 s-1 → mm day-1
pr_yr.pr.attrs["units"] = "mm/day"

# Use a more recent baseline for precipitation (1981–2010 is the WMO standard)
pr_anom = calc_anom(pr_yr, base_start=1981, base_end=2010)
pr_anom.pr.attrs["units"] = "mm/day anomaly"

#### Step 5b: Coefficient of variation — normalised spread comparison

The **coefficient of variation** (CV = std / |mean|) normalises inter-model spread by the signal magnitude, making it comparable across variables with different units and scales. Higher CV = more model disagreement relative to the projected change.

In [ ]:
#  compute CV (std / |mean|) for each variable over end-of-century
# requires: xy_anom (temperature) and pr_anom (precipitation) from the notebook
# both already computed in Step 5

eoc_slice = slice(2071, 2100)


def compute_cv(da, dim="simulation"):
    """Coefficient of variation: inter-model std / |inter-model mean|."""
    mean = da.sel(time=eoc_slice).mean(dim=["time", dim])
    std = da.sel(time=eoc_slice).std(dim=dim).mean(dim="time")
    # floor the mean to avoid division near zero
    return float((std / mean.where(np.abs(mean) > 0.05, other=0.05)).mean())


cv_tas = compute_cv(xy_anom.tas)
cv_pr = compute_cv(pr_anom.pr)

#  if you have wind speed loaded, add it here
# cv_wind = compute_cv(wind_anom.sfcWind)

variables = ["Max temperature", "Precipitation"]
cv_values = [cv_tas, cv_pr]
colors = ["#4472A8", "#C0664A"]

#  plot
fig, ax = plt.subplots(figsize=(6, 2.5))

bars = ax.barh(
    variables,
    cv_values,
    color=colors,
    height=0.45,
    edgecolor="white",
    linewidth=0.5,
)

# value labels
for bar, val in zip(bars, cv_values):
    ax.text(
        val + 0.003,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        ha="left",
        fontsize=10,
        color="#444",
    )

# threshold line at CV = 1 (spread equals signal)
ax.axvline(1.0, color="#888", lw=1, ls="dashed", alpha=0.6)
ax.text(
    1.02,
    0.02,
    "CV = 1\n(spread = signal)",
    transform=ax.get_yaxis_transform(),
    fontsize=8,
    color="#888",
    va="bottom",
)

ax.set_xlabel(
    "Coefficient of variation (inter-model std / |mean|)", fontsize=10)
ax.set_title(
    "Inter-model spread by variable — end of century, SSP3-7.0\n"
    "Higher CV = more model disagreement relative to projected change",
    fontsize=10,
    pad=10,
)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=10)
ax.set_xlim(left=0)
ax.grid(axis="x", alpha=0.2, lw=0.6)

plt.tight_layout()

fig.savefig("figures/static/variable_cv.png", dpi=150)

Precipitation CV is substantially higher than temperature CV across most of California — meaning that the choice of model matters far more for precipitation-dependent applications (water supply, flood risk) than for heat-related ones. Regions where the precipitation anomaly mean is near zero (models split between wetting and drying) produce very high CV values, which is itself a signal of deep model disagreement.